In [ ]:
import os
import sys
import jax
jax.config.update("jax_enable_x64", True)

import numpy as np

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.config import ExperimentConfig
from src.physics.operators import build_two_qubit_base, assemble_H0, build_true_H0
from src.analysis.metrics import cz_process_fidelity, cz_comp_block, bell_from_cz_block, cz_process_fidelity
from src.devices.qutip_device import HiddenDevice
from src.experiments.process_tomography import run_process_tomography

config = ExperimentConfig.load("configs/two_qubit_CZ_zz.yaml")
dt = config.pulse.dt_ns
hw = config.hardware

# Twin: fitted parameters from notebook 01
H_anharm, H_int_base, Hz_control = build_two_qubit_base(levels=hw.levels)
cal_g, cal_d, cal_a = np.load(config.artifact_path("cal_params.npy"))
H0_cal = assemble_H0(H_anharm, H_int_base, Hz_control, cal_g, cal_d, cal_a)

pulse = np.load(config.artifact_path("cal_pulse_rad.npy"))

In [ ]:
H0_true, _ = build_true_H0(config)   # exact device Hamiltonian incl. unmodeled ZZ, no decoherence

rows = {"twin": H0_cal, "oracle device": H0_true}
for name, H0 in rows.items():
    U = cz_comp_block(pulse, dt, H0, Hz_control)
    r = cz_process_fidelity(U)
    print(f"{name:14s} F_avg={r['F_avg']*100:.3f}%  F_pro={r['F_pro']*100:.3f}%  "
          f"leakage={r['leakage']*100:.3f}%  Bell (ideal 1q gates)={bell_from_cz_block(U)*100:.3f}%")

In [ ]:
# Validation Test: Noiseless device (T1=0, T2=0, n_shots=None) vs. Oracle
dev_noiseless = HiddenDevice.from_config(config, t1_ns=0.0, t2_ns=0.0)
choi_exact, _ = run_process_tomography(dev_noiseless, pulse, n_shots=None)
r_qpt_exact = cz_process_fidelity(choi_exact)

# Re-compute oracle for direct comparison
U_oracle = cz_comp_block(pulse, dt, H0_true, Hz_control)
r_oracle = cz_process_fidelity(U_oracle)

print("=== 1. QPT Method Validation (Noiseless vs. Oracle) ===")
print(f"Oracle (from H0_true) : F_avg={r_oracle['F_avg']*100:.4f}%  F_pro={r_oracle['F_pro']*100:.4f}%  leakage={r_oracle['leakage']*100:.4f}%")
print(f"QPT (noiseless data)  : F_avg={r_qpt_exact['F_avg']*100:.4f}%  F_pro={r_qpt_exact['F_pro']*100:.4f}%  leakage={r_qpt_exact['leakage']*100:.4f}%")
print(f"Max absolute diff     : dF_pro={abs(r_oracle['F_pro'] - r_qpt_exact['F_pro']):.2e}, dLeak={abs(r_oracle['leakage'] - r_qpt_exact['leakage']):.2e}\n")

# Open-System Device (with T1, T2 decoherence, infinite shots)
dev_noisy = HiddenDevice.from_config(config)
choi_noisy, _ = run_process_tomography(dev_noisy, pulse, n_shots=None)
r_qpt_noisy = cz_process_fidelity(choi_noisy)

print("=== 2. Open-System Device Process Tomography (With T1/T2 Decoherence) ===")
print(f"QPT (T1/T2, inf shots): F_avg={r_qpt_noisy['F_avg']*100:.3f}%  F_pro={r_qpt_noisy['F_pro']*100:.3f}%  leakage={r_qpt_noisy['leakage']*100:.3f}%")
print(f"Decoherence penalty   : dF_avg={(r_qpt_exact['F_avg'] - r_qpt_noisy['F_avg'])*100:.3f}%  dF_pro={(r_qpt_exact['F_pro'] - r_qpt_noisy['F_pro'])*100:.3f}%\n")

# Shot-Noise Convergence Check
print("=== 3. Shot-Noise Convergence (10 independent trials per n_shots) ===")
for n_shots in [500, 2000, 10000]:
    f_pros = []
    for trial in range(10):
        choi_shot, _ = run_process_tomography(dev_noisy, pulse, n_shots=n_shots, seed=100 + trial)
        f_pros.append(cz_process_fidelity(choi_shot)["F_pro"] * 100)
    print(f"n_shots={n_shots:5d} -> F_pro = {np.mean(f_pros):.3f}% ± {np.std(f_pros):.3f}%")